# Scala en Binder: prueba mínima

**Objetivo:** comprobar el kernel Scala, el estado entre celdas y las operaciones que utilizaremos en el tutorial.

Este notebook es un esqueleto técnico resuelto, no la exposición final. Ejecuta de arriba hacia abajo con **Shift + Enter**, o usa **Kernel → Restart Kernel and Run All Cells**. Selecciona **Scala 3.9**.

Caso: un catálogo de productos. Calcularemos el precio de comprar **una unidad de cada producto disponible**, con un descuento común. Importes en pesos colombianos (COP).

**Resultado esperado con 10 %:** 315000 COP y `PRUEBA COMPLETA: OK` al final.

Binder es temporal: descarga el notebook si quieres conservar tus cambios. Guardarlo aquí no actualiza GitHub.

## 1. Primera ejecución — integrante 1

Estas dos celdas prueban que una definición de Scala permanece disponible para la siguiente. La función usa sintaxis con indentación de Scala 3.

In [ ]:
val mensaje = "Scala funciona"
val numeroDePrueba = 21

def duplicar(n: Int): Int =
  n * 2

println(mensaje)

In [ ]:
assert(duplicar(numeroDePrueba) == 42)
println("ESTADO ENTRE CELDAS: OK")

## 2. Función sobre un precio — integrante 2

Contrato provisional: precio válido no negativo y porcentaje entero entre 0 y 100. `BigDecimal` se construye con texto para representar los decimales del ejemplo. La política de redondeo comercial queda fuera de esta prueba.

`porcentajeDescuento` será utilizado por los bloques posteriores.

In [ ]:
val porcentajeDescuento = 10

def aplicarDescuento(precio: BigDecimal, porcentaje: Int): BigDecimal =
  precio * (BigDecimal(100) - BigDecimal(porcentaje)) / BigDecimal(100)

assert(porcentajeDescuento >= 0 && porcentajeDescuento <= 100)
assert(aplicarDescuento(BigDecimal("100000"), 10) == BigDecimal("90000"))
assert(aplicarDescuento(BigDecimal("100000"), 0) == BigDecimal("100000"))
assert(aplicarDescuento(BigDecimal("100000"), 100) == BigDecimal(0))
println("FUNCION DE DESCUENTO: OK")

## 3. Modelo y catálogo inmutable — integrante 3

Campos acordados como punto de partida: identificador, nombre, precio y disponibilidad. Todos los precios son válidos; el programa no procesa textos mal formados.

El mouse no está disponible y no se incluirá en el total.

In [ ]:
case class Producto(id: Int, nombre: String, precio: BigDecimal, disponible: Boolean)

val productos = List(
  Producto(1, "Teclado", BigDecimal("100000"), true),
  Producto(2, "Audifonos", BigDecimal("200000"), true),
  Producto(3, "Mouse", BigDecimal("80000"), false),
  Producto(4, "Soporte", BigDecimal("50000"), true)
)

assert(productos.size == 4)
productos.foreach(println)

In [ ]:
val tecladoOriginal = productos.head
val tecladoEnOferta = tecladoOriginal.copy(
  precio = aplicarDescuento(tecladoOriginal.precio, porcentajeDescuento)
)

assert(tecladoOriginal.precio == BigDecimal("100000"))
assert(tecladoEnOferta.precio == aplicarDescuento(BigDecimal("100000"), porcentajeDescuento))
println(s"Original: ${tecladoOriginal.precio}; copia: ${tecladoEnOferta.precio}")

## 4. Seleccionar, transformar y acumular — integrante 4

Cada paso usa el resultado anterior. Las lambdas llevan parámetros con nombre para facilitar su lectura. El cálculo conserva los productos originales; la impresión se usa únicamente para observar el resultado.

In [ ]:
val disponibles = productos.filter(producto => producto.disponible)
assert(disponibles.map(producto => producto.id) == List(1, 2, 4))
println(s"Disponibles: ${disponibles.map(producto => producto.nombre)}")

In [ ]:
val preciosConDescuento = disponibles.map(
  producto => aplicarDescuento(producto.precio, porcentajeDescuento)
)
val factorEsperado = (BigDecimal(100) - BigDecimal(porcentajeDescuento)) / BigDecimal(100)
val preciosEsperados = List("100000", "200000", "50000").map(
  texto => BigDecimal(texto) * factorEsperado
)
assert(preciosConDescuento == preciosEsperados)
println(s"Precios con descuento: $preciosConDescuento")

In [ ]:
val total = preciosConDescuento.foldLeft(BigDecimal(0))(
  (acumulado, precio) => acumulado + precio
)
assert(total == BigDecimal("350000") * factorEsperado)
assert(productos.head.precio == BigDecimal("100000"))
println(s"Total con $porcentajeDescuento % de descuento: $total COP")

In [ ]:
val catalogoVacio = List.empty[Producto]
val totalVacio = catalogoVacio
  .filter(producto => producto.disponible)
  .map(producto => aplicarDescuento(producto.precio, porcentajeDescuento))
  .foldLeft(BigDecimal(0))((acumulado, precio) => acumulado + precio)

assert(totalVacio == BigDecimal(0))
println("CATALOGO VACIO: OK (total 0)")

## 5. Búsqueda con presencia o ausencia — integrante 5

La búsqueda devuelve `Option[Producto]`. Consultar por identificador es independiente de que el producto esté disponible.

In [ ]:
def buscarProducto(id: Int): Option[Producto] =
  productos.find(producto => producto.id == id)

def describirResultado(resultado: Option[Producto]): String =
  resultado match
    case Some(producto) => s"Encontrado: ${producto.nombre}"
    case None => "Producto no encontrado"

assert(buscarProducto(1).contains(tecladoOriginal))
assert(buscarProducto(999).isEmpty)
assert(describirResultado(buscarProducto(999)) == "Producto no encontrado")
println(describirResultado(buscarProducto(1)))
println(describirResultado(buscarProducto(999)))

In [ ]:
println("PRUEBA COMPLETA: OK")

## Comprobación manual y siguiente paso

1. Cambiar `porcentajeDescuento` de 10 a 20 y vuelve a ejecutar desde la celda de esa definición hasta el final. El total debe ser **280000 COP**.
2. Restaurar 10, reinicia el kernel y ejecuta todo en orden. Debe volver a dar **315000 COP**.
3. Descargar el notebook para conservar tu trabajo.
